# Module 1 · Foundations
**Xenoi Biotech · DNA-Seq (Variant Analysis) Workshop** · Week 1 · Days 1–5 · about 1–2 hours per day

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anjalianjali1234/dna-seq-variant-colab/blob/main/notebooks/M1_Foundations.ipynb)

**By the end of this module:** you can use the Linux command line, compare sequencing designs, read FASTQ/BAM/VCF files and make a read-QC decision.

**Workshop question:** can different variant callers recover the same small DNA changes in a real human sample (HG001 / NA12878), and how do we know which calls to trust?

**How to use this notebook:** run the cells from top to bottom. Read each explanation before running the code under it. Cells marked ✍️ are exercises: write your answers in your lab notebook.

## Session setup (run every time you open Colab)
Colab gives you a fresh computer each session, so this cell installs the tools and downloads the data. The first run takes about **3–5 minutes**; go through the reading below while it works.

In [ ]:
import os
%cd /content
if not os.path.exists("dna-seq-variant-colab"):
    !git clone -q https://github.com/anjalianjali1234/dna-seq-variant-colab.git
%cd /content/dna-seq-variant-colab
!bash setup/colab_setup.sh
os.environ["PATH"] = "/content/env/bin:" + os.environ["PATH"]
!samtools --version | head -1 && gatk --version 2>/dev/null | head -1

> 💾 **Optional:** to keep your outputs between sessions, mount Google Drive and copy `run/` there at the end:
> `from google.colab import drive; drive.mount('/content/drive')` then `!cp -r run /content/drive/MyDrive/dna-seq-run`

## Day 1 · Linux and command-line basics
Bioinformatics tools run from the command line. In Colab, a cell that starts with `%%bash` runs as a Linux terminal. Each `%%bash` cell starts fresh, so use full paths or `cd` at the top of the cell.

| Command | What it does |
|---|---|
| `pwd` | where am I? |
| `ls -lh` | list files with sizes |
| `cd folder` | move into a folder |
| `head`, `tail`, `less` | look inside a file |
| `wc -l` | count lines |
| `grep` | search text |
| `|` | send one command's output into the next |

In [ ]:
%%bash
cd /content/dna-seq-variant-colab
pwd
ls -lh
ls -lh run/input

In [ ]:
%%bash
cd /content/dna-seq-variant-colab
cat data/sources.tsv | cut -f 1,2
grep -c "" data/sources.tsv   # number of lines, including the header

> ✍️ **Your turn**
>
> 1. How many input files does the workshop download? Which is the largest?
> 2. Use `mkdir`, `cp` and `ls` to make a folder `notebook/` and copy `data/sources.tsv` into it.
> 3. Explain in one sentence what the `md5` column is for.

## Day 2 · DNA sequencing designs
| Design | What it covers | Typical use |
|---|---|---|
| **Targeted panel** | tens to hundreds of known genes, very deep | known disease genes, tumours |
| **Whole exome (WES)** | protein-coding exons (~1–2% of the genome) | rare disease, coding variants |
| **Whole genome (WGS)** | almost the entire genome, including introns and intergenic DNA | structural and non-coding variants, research |

This workshop uses a small region of a **whole-genome** sample, HG001 (NA12878), one of the best-characterised human genomes. The Genome in a Bottle consortium (NIST) has published a high-confidence **truth set** of its variants, so we can check our own calls against it.

> ✍️ **Your turn**
>
> 1. A patient may carry a variant in an **intron** of a gene. Which designs could detect it, and which could not? Why?
> 2. Which design would you choose for a study of 500 patients with a suspected inherited heart condition and a limited budget? Justify it.
> 3. What is the main advantage of using a sample with a known truth set while learning?

## Day 3 · File formats: FASTQ, BAM, VCF and reference files
- **FASTA** (`ref.fa`): the reference genome sequence.
- **FASTQ**: raw reads, 4 lines per read (name, sequence, `+`, quality).
- **BAM**: reads aligned to the reference (binary; read with `samtools view`).
- **VCF**: variant calls, one line per variant (chromosome, position, REF, ALT, quality, genotype).
- **BED**: genomic intervals (0-based start, end).

In [ ]:
%%bash
cd /content/dna-seq-variant-colab
head -c 300 run/input/ref.fa; echo
samtools view -H run/input/reads.bam | head -5
samtools view run/input/reads.bam | head -2 | cut -f 1-9
bcftools view -H run/input/truth.vcf.gz | head -3
head -3 run/input/confident.bed

> ✍️ **Your turn**
>
> 1. In the BAM lines, which columns give the chromosome, position, mapping quality and CIGAR string?
> 2. In the truth VCF, pick one variant: what are its REF and ALT alleles and its genotype?
> 3. Why does the BED file start counting from 0 while the VCF starts from 1?

## Day 4 · Recover read pairs and inspect them
The data arrive as an already-aligned regional BAM. We convert it back to paired FASTQ files so we can do every step ourselves.

In [ ]:
%%bash
cd /content/dna-seq-variant-colab
samtools collate -u -O run/input/reads.bam | samtools fastq -F 0x900 -n \
    -1 run/input/R1.fq.gz -2 run/input/R2.fq.gz -s run/input/orphans.fq.gz -0 /dev/null - 2>/dev/null
zcat run/input/R1.fq.gz | head -4
echo "Read pairs in R1:"; zcat run/input/R1.fq.gz | awk 'END {print NR/4}'
echo "Read pairs in R2:"; zcat run/input/R2.fq.gz | awk 'END {print NR/4}'

> ✍️ **Your turn**
>
> 1. Label the four lines of the first read. How many bases and how many quality characters does it have?
> 2. Do R1 and R2 contain the same number of reads? Why must they?
> 3. A quality of Q30 means an error probability of 0.001. What does Q20 mean?

## Day 5 · Read quality control with fastp

In [ ]:
%%bash
cd /content/dna-seq-variant-colab
mkdir -p run/qc
fastp -i run/input/R1.fq.gz -I run/input/R2.fq.gz -o run/qc/R1.fq.gz -O run/qc/R2.fq.gz \
    --thread 2 --dont_eval_duplication --html run/qc/fastp.html --json run/qc/fastp.json 2>/dev/null
echo "QC done"

In [ ]:
import json
q = json.load(open("run/qc/fastp.json"))
for stage in ("before_filtering", "after_filtering"):
    s = q["summary"][stage]
    print(f"{stage:17s} reads={s['total_reads']:,}  Q30={s['q30_rate']:.3f}  mean length={s['read1_mean_length']}")
print(q["filtering_result"])

In [ ]:
from IPython.display import IFrame
IFrame("run/qc/fastp.html", width="100%", height=500)

> ✍️ **Your turn**
>
> **Module 1 deliverable:** write a 5-line QC decision:
> - reads before and after filtering, and the % retained
> - the Q30 rate after filtering
> - one feature of the per-position quality plot
> - what stricter trimming would gain and what it would cost
> - your decision: proceed to alignment, or re-trim?